# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Accessing mental health therapy can be navigated through several paths depending on your insurance status, budget, and specific needs. Below is a comprehensive guide to finding and starting therapy.

### **1. Immediate Crisis Resources**
If you or someone else is in danger or needs immediate help, do not wait for an appointment. Use these free, confidential resources available 24/7:
*   **988 Suicide & Crisis Lifeline:** Call or text **988** (US) for free, confidential support for any mental health distress.
*   **Crisis Text Line:** Text **HOME** to **741741** to connect with a crisis counselor.
*   **Veterans Crisis Line:** Dial **988, then press 1** or text **838255**.
*   **The Trevor Project (LGBTQ+ Youth):** Text **START** to **678-678** or call **1-866-488-7386**.
*   **National Domestic Violence Hotline:** Text **START** to **88788** or call **1-800-799-SAFE (7233)**.

---

### **2. Using Health Insurance (The Most Common Route)**
If you have health insurance, it often covers outpatient mental health services.
*   **Check Your Card:** Look for a "Behavioral Health" or "Mental Health" phone number on the back of your insurance card. If not listed, call "Customer Service."
*   **Verify Your Benefits:** Ask the representative these specific questions:
    *   *Do I have coverage for outpatient mental health office visits?*
    *   *What is my co-pay per session?*
    *   *Do I have a deductible I need to meet first?*
    *   *Do I need a referral from my primary care doctor?*
*   **Find a Provider:** Ask your insurance for a list of in-network providers or use their online portal to search for therapists near you.

### **3. Using Employee Assistance Programs (EAP)**
Many employers offer an **Employee Assistance Program (EAP)**, which provides a limited number of **free counseling sessions** (typically 3–10) per issue per year.
*   **How to Access:** Contact your HR department or look at your benefits portal. This is a separate benefit from your health insurance and is strictly confidential—your employer will not know you used it.
*   **Best For:** Short-term stress, work-life balance issues, grief, or immediate support while looking for a long-term therapist.

### **4. Finding a Therapist (Directories & Search Tools)**
You can search for therapists by location, specialty, insurance, and price using these reputable directories:
*   **Psychology Today:** The most widely used directory. You can filter by "Issues" (e.g., anxiety, trauma), "Insurance," and "Price."
*   **GoodTherapy:** Focuses on therapists committed to ethical, non-pathologizing practices.
*   **Mental Health Match:** A matching tool that asks you questions to pair you with compatible therapists.
*   **Open Path Collective:** A directory specifically for **low-cost therapy** ($40–$70/session) for those without insurance or with high deductibles. (Requires a one-time lifetime membership fee of ~$65).

### **5. Low-Cost & Free Options (No Insurance)**
If you do not have insurance or cannot afford standard rates ($100–$200+ per session), consider these options:
*   **University Training Clinics:** Universities with graduate psychology programs often run public clinics where you see a student therapist (supervised by a licensed professional) for very low fees (often sliding scale based on income).
*   **Community Health Centers:** Federally Qualified Health Centers (FQHCs) often provide therapy on a sliding fee scale (pay based on what you earn). Search for one at **[Find A Health Center](https://findahealthcenter.hrsa.gov/)**.
*   **Ask for "Sliding Scale":** Many private therapists reserve a few spots for clients at a reduced rate. It is always worth asking, *"Do you offer a sliding scale option?"*

### **6. Choosing the Right Type of Professional**
Understanding the "alphabet soup" of credentials can help you find the right fit:
*   **Psychiatrist (MD/DO):** Medical doctors who primarily **prescribe medication**. They usually do less "talk therapy."
*   **Psychologist (PhD/PsyD):** Have a doctorate degree; provide therapy and often specialize in psychological testing/assessments.
*   **LCSW (Licensed Clinical Social Worker):** Trained in talk therapy with an emphasis on how your environment/life situation affects your mental health.
*   **LMFT (Licensed Marriage & Family Therapist):** Specialize in relationships and family dynamics (even for individual therapy).
*   **LPC / LMHC (Licensed Professional Counselor):** Trained in general counseling and talk therapy for a wide range of issues.

### **7. What to Expect in the First Session (Intake)**
Your first appointment is usually an "intake" session.
*   **Paperwork:** You will sign consent forms and privacy policies (HIPAA).
*   **History:** The therapist will ask about your personal history, current symptoms, and why you are seeking therapy *now*.
*   **Goal Setting:** You will discuss what you want to get out of therapy (e.g., "I want to feel less anxious at work" or "I want to communicate better with my partner").
*   **Fit:** It is okay to "shop around." If you don't feel comfortable after the first few sessions, it is perfectly acceptable to find a different therapist. The relationship is the most important factor for success.

Based on the insurance document provided, here's how you can access mental health therapy under this Anthem Blue Cross HDHP plan:

## **Coverage for Mental Health Services**

Your plan covers **outpatient and inpatient mental health, behavioral health, and substance abuse services** with the following costs:

### **Outpatient Mental Health Services (After Deductible)**
- **In-Network Provider:** 10% coinsurance
- **Out-of-Network Provider:** 30% coinsurance

### **Inpatient Mental Health Services (After Deductible)**
- **In-Network Provider:** 10% coinsurance
- **Out-of-Network Provider:** 30% coinsurance
- **Inpatient Physician Fees:** 10% in-network / 30% out-of-network

## **Important Coverage Details**

1. **No Referral Required:** You can see a mental health specialist without needing a referral from your primary care doctor.

2. **Deductible Applies:** You must meet your annual deductible before coinsurance costs apply:
   - \\$1,700 (individual) or \\$3,400 (family) for in-network providers
   - \\$3,400 (individual) or \\$6,800 (family) for out-of-network providers

3. **Network Providers:** To pay the least, use in-network providers. Your plan uses:
   - **NY Blue Access PPO** (if in NY, excluding Suffolk County)
   - **GA Blue Open Access POS** (if in GA)
   - **Blue Card PPO** (all other areas)

## **Finding a Provider**

See the network provider list at **includedhealth.com/google** or call **(855) 431-5540** to find in-network mental health providers in your area.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
